# Final Assignment: Tesla & GameStop - Giá cổ phiếu vs Doanh thu

In [1]:
!pip install yfinance bs4 lxml html5lib nbformat plotly

In [2]:
import yfinance as yf
import pandas as pd
import requests
from io import StringIO
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def make_graph(stock_data, revenue_data, stock):
    stock_data = stock_data.copy(); revenue_data = revenue_data.copy()
    stock_data['Date'] = pd.to_datetime(stock_data['Date']).dt.tz_localize(None)
    revenue_data['Date'] = pd.to_datetime(revenue_data['Date'])
    s = stock_data[stock_data.Date <= '2021-06-14']
    r = revenue_data[revenue_data.Date <= '2021-04-30']
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("Historical Share Price", "Historical Revenue"),
                        vertical_spacing=.3)
    fig.add_trace(go.Scatter(x=s.Date, y=s.Close.astype('float'), name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=r.Date, y=r.Revenue.astype('float'), name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock, xaxis_rangeslider_visible=True)
    fig.show()

# Hàm dùng chung để lấy bảng doanh thu theo quý từ trang HTML
def get_revenue(url, match):
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
    table = pd.read_html(StringIO(html), match=match)[0]
    table = table.iloc[:, :2]
    table.columns = ["Date", "Revenue"]
    table["Revenue"] = table["Revenue"].astype(str).str.replace(r"[$,]", "", regex=True)
    table = table[(table["Revenue"] != "") & (table["Revenue"] != "nan")]
    return table.dropna().reset_index(drop=True)

## Question 1 - Tesla stock (yfinance)

In [3]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


## Question 2 - Tesla revenue (web scraping)

In [4]:
# Nếu đề bài (lab) cung cấp URL khác, hãy thay vào đây
url_tesla = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
tesla_revenue = get_revenue(url_tesla, "Tesla Quarterly Revenue")
tesla_revenue.tail()

,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
51,2009-09-30,46
52,2009-06-30,27


## Question 3 - GameStop stock (yfinance)

In [5]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620128,1.693350,1.603296,1.691666,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683250,1.687458,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615920,1.662209,1.603296,1.662209,6892800,0.0,0.0


## Question 4 - GameStop revenue (web scraping)

In [6]:
url_gme = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
gme_revenue = get_revenue(url_gme, "GameStop Quarterly Revenue")
gme_revenue.tail()

,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


## Question 5 - Tesla dashboard

In [7]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

## Question 6 - GameStop dashboard

In [8]:
make_graph(gme_data, gme_revenue, 'GameStop')